In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

PROJECT_ROOT = Path(
    r"Z:\Projects\monsoon-postprocessing"
)

BOUNDARY_FILE = (
    PROJECT_ROOT
    / "data"
    / "boundaries"
    / "india_adm2.geojson"
)

FORECAST_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "district_regime_corrected_forecast_20180727_20180731.csv"
)

DEPLOYMENT_BOUNDARY_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "india_districts_simplified.geojson"
)

districts = gpd.read_file(BOUNDARY_FILE)

deployment_districts = districts[
    [
        "shapeID",
        "shapeName",
        "geometry",
    ]
].copy()

deployment_districts["geometry"] = (
    deployment_districts.geometry.simplify(
        tolerance=0.02,
        preserve_topology=True,
    )
)

deployment_districts.to_file(
    DEPLOYMENT_BOUNDARY_FILE,
    driver="GeoJSON",
)

forecast = pd.read_csv(FORECAST_FILE)

print(
    "Boundary size:",
    round(
        DEPLOYMENT_BOUNDARY_FILE.stat().st_size
        / 1_000_000,
        2,
    ),
    "MB",
)

print(
    "Forecast size:",
    round(
        FORECAST_FILE.stat().st_size
        / 1_000_000,
        2,
    ),
    "MB",
)

print(
    "Boundary districts:",
    len(deployment_districts),
)

print(
    "Forecast rows:",
    len(forecast),
)

Boundary size: 0.75 MB
Forecast size: 0.8 MB
Boundary districts: 735
Forecast rows: 3675


In [1]:
from pathlib import Path

import pandas as pd
import xarray as xr

# Locate the project root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

CORRECTION_FILE = (
    ROOT / "data" / "processed" /
    "july2018_regime_correction_predictions.nc"
)

LOOKUP_FILE = (
    ROOT / "data" / "processed" /
    "complete_grid_district_lookup.csv"
)

DEPLOYMENT_FILE = (
    ROOT / "data" / "processed" /
    "district_rainfall_forecast_20180727_20180731.csv"
)

LARGE_GEOJSON_FILE = (
    ROOT / "data" / "processed" /
    "district_regime_corrected_forecast.geojson"
)

files = {
    "Correction NetCDF": CORRECTION_FILE,
    "Grid-district lookup": LOOKUP_FILE,
    "Current deployment CSV": DEPLOYMENT_FILE,
    "Original combined GeoJSON": LARGE_GEOJSON_FILE,
}

print("PROJECT ROOT")
print(ROOT)

print("\nFILE CHECK")
for label, file in files.items():
    size_mb = (
        round(file.stat().st_size / (1024 * 1024), 3)
        if file.exists()
        else None
    )

    print(
        f"{label}: exists={file.exists()}, "
        f"size_mb={size_mb}, path={file}"
    )

if CORRECTION_FILE.exists():
    with xr.open_dataset(CORRECTION_FILE) as correction_ds:
        print("\nCORRECTION DATASET")
        print(correction_ds)

        print("\nCorrection variables:")
        print(list(correction_ds.data_vars))

        print("\nCorrection coordinates:")
        print(list(correction_ds.coords))

if LOOKUP_FILE.exists():
    lookup = pd.read_csv(LOOKUP_FILE)

    print("\nLOOKUP TABLE")
    print("Rows:", len(lookup))
    print("Columns:")
    print(lookup.columns.tolist())

    display(lookup.head())

if DEPLOYMENT_FILE.exists():
    deployment = pd.read_csv(
        DEPLOYMENT_FILE,
        parse_dates=["date"],
    )

    print("\nCURRENT DEPLOYMENT TABLE")
    print("Rows:", len(deployment))
    print("Districts:", deployment["shapeID"].nunique())
    print("Dates:", deployment["date"].nunique())

    print("\nDeployment columns:")
    print(deployment.columns.tolist())

    required_columns = [
        "gefs_mean_mm",
        "global_corrected_mean_mm",
        "regime_corrected_mean_mm",
        "observed_mean_mm",
        "heavy_probability_max",
    ]

    validation = pd.DataFrame(
        {
            "column": required_columns,
            "available": [
                column in deployment.columns
                for column in required_columns
            ],
            "non_null_values": [
                (
                    int(deployment[column].notna().sum())
                    if column in deployment.columns
                    else 0
                )
                for column in required_columns
            ],
        }
    )

    display(validation)

PROJECT ROOT
Z:\Projects\monsoon-postprocessing

FILE CHECK
Correction NetCDF: exists=True, size_mb=5.948, path=Z:\Projects\monsoon-postprocessing\data\processed\july2018_regime_correction_predictions.nc
Grid-district lookup: exists=True, size_mb=0.271, path=Z:\Projects\monsoon-postprocessing\data\processed\complete_grid_district_lookup.csv
Current deployment CSV: exists=True, size_mb=0.785, path=Z:\Projects\monsoon-postprocessing\data\processed\district_rainfall_forecast_20180727_20180731.csv
Original combined GeoJSON: exists=True, size_mb=194.12, path=Z:\Projects\monsoon-postprocessing\data\processed\district_regime_corrected_forecast.geojson

CORRECTION DATASET
<xarray.Dataset> Size: 8MB
Dimensions:                    (date: 31, latitude: 129, longitude: 121)
Coordinates:
  * date                       (date) datetime64[ns] 248B 2018-07-01 ... 2018...
  * latitude                   (latitude) float64 1kB 6.0 6.25 ... 37.75 38.0
  * longitude                  (longitude) float64 968B

,latitude,longitude,shapeName,shapeID,mapping_method
0,7.00,93.75,Nicobars,76128533B33103505211400,contained_grid
1,8.00,93.50,Nicobars,76128533B33103505211400,contained_grid
2,8.25,77.25,Kanniyakumari,76128533B53796141759805,contained_grid
3,8.25,77.50,Kanniyakumari,76128533B53796141759805,contained_grid
4,8.25,77.75,Tirunelveli,76128533B52639434516056,contained_grid



CURRENT DEPLOYMENT TABLE
Rows: 3675
Districts: 735
Dates: 5

Deployment columns:
['shapeID', 'shapeName', 'date', 'mapping_method', 'gefs_mean_mm', 'gefs_max_mm', 'observed_mean_mm', 'observed_max_mm', 'heavy_probability_mean', 'heavy_probability_max', 'very_heavy_probability_max', 'grid_cells', 'risk_level', 'very_heavy_experimental', 'correction_raw_mean_mm', 'global_corrected_mean_mm', 'global_corrected_max_mm', 'regime_corrected_mean_mm', 'regime_corrected_max_mm', 'regime_adjustment_mm', 'global_adjustment_mm']


,column,available,non_null_values
0,gefs_mean_mm,True,3675
1,global_corrected_mean_mm,True,3675
2,regime_corrected_mean_mm,True,3675
3,observed_mean_mm,True,3675
4,heavy_probability_max,True,3675


In [2]:
import numpy as np
import pandas as pd
import xarray as xr

TEST_START = pd.Timestamp("2018-07-27")
TEST_END = pd.Timestamp("2018-07-31")
TOLERANCE = 1e-4

# Read the correction dataset into a grid-level table.
with xr.open_dataset(CORRECTION_FILE) as ds:
    correction_grid = (
        ds[
            [
                "raw_gefs_rainfall",
                "global_corrected_rainfall",
                "regime_corrected_rainfall",
                "imerg_rainfall",
            ]
        ]
        .sel(date=slice(TEST_START, TEST_END))
        .load()
        .to_dataframe()
        .reset_index()
    )

lookup = pd.read_csv(LOOKUP_FILE)

# Round coordinates to prevent floating-point merge mismatches.
for table in [correction_grid, lookup]:
    table["latitude_key"] = table["latitude"].round(5)
    table["longitude_key"] = table["longitude"].round(5)

mapped_grid = lookup.merge(
    correction_grid,
    on=["latitude_key", "longitude_key"],
    how="left",
    validate="many_to_many",
    suffixes=("_lookup", "_grid"),
)

mapped_grid["date"] = pd.to_datetime(mapped_grid["date"])

# Recalculate district statistics directly from the NetCDF.
recalculated = (
    mapped_grid
    .groupby(
        ["shapeID", "date"],
        as_index=False,
    )
    .agg(
        recalculated_raw_mean=(
            "raw_gefs_rainfall",
            "mean",
        ),
        recalculated_global_mean=(
            "global_corrected_rainfall",
            "mean",
        ),
        recalculated_global_max=(
            "global_corrected_rainfall",
            "max",
        ),
        recalculated_regime_mean=(
            "regime_corrected_rainfall",
            "mean",
        ),
        recalculated_regime_max=(
            "regime_corrected_rainfall",
            "max",
        ),
        recalculated_observed_mean=(
            "imerg_rainfall",
            "mean",
        ),
        recalculated_observed_max=(
            "imerg_rainfall",
            "max",
        ),
        recalculated_grid_cells=(
            "raw_gefs_rainfall",
            "count",
        ),
    )
)

deployment = pd.read_csv(
    DEPLOYMENT_FILE,
    parse_dates=["date"],
)

comparison = deployment.merge(
    recalculated,
    on=["shapeID", "date"],
    how="outer",
    validate="one_to_one",
    indicator=True,
)

comparisons = {
    "correction_raw_mean_mm": "recalculated_raw_mean",
    "global_corrected_mean_mm": "recalculated_global_mean",
    "global_corrected_max_mm": "recalculated_global_max",
    "regime_corrected_mean_mm": "recalculated_regime_mean",
    "regime_corrected_max_mm": "recalculated_regime_max",
    "observed_mean_mm": "recalculated_observed_mean",
    "observed_max_mm": "recalculated_observed_max",
    "grid_cells": "recalculated_grid_cells",
}

verification_rows = []

for saved_column, recalculated_column in comparisons.items():
    difference = (
        pd.to_numeric(
            comparison[saved_column],
            errors="coerce",
        )
        - pd.to_numeric(
            comparison[recalculated_column],
            errors="coerce",
        )
    ).abs()

    verification_rows.append(
        {
            "saved_column": saved_column,
            "recalculated_column": recalculated_column,
            "compared_values": int(difference.notna().sum()),
            "missing_comparisons": int(difference.isna().sum()),
            "maximum_absolute_difference": difference.max(),
            "mean_absolute_difference": difference.mean(),
            "values_above_tolerance": int(
                (difference > TOLERANCE).sum()
            ),
            "passed": bool(
                difference.dropna().le(TOLERANCE).all()
            ),
        }
    )

verification_summary = pd.DataFrame(verification_rows)

print("Mapped grid records:", len(mapped_grid))
print("Mapped records without rainfall:", int(
    mapped_grid["raw_gefs_rainfall"].isna().sum()
))
print("Recalculated district-date rows:", len(recalculated))
print("Deployment district-date rows:", len(deployment))

print("\nMerge status:")
print(comparison["_merge"].value_counts())

display(verification_summary)

all_rows_matched = comparison["_merge"].eq("both").all()
all_metrics_passed = verification_summary["passed"].all()

print("\nVALIDATION RESULT")
print("All district-date rows matched:", all_rows_matched)
print("All metric comparisons passed:", all_metrics_passed)

if all_rows_matched and all_metrics_passed:
    print("Stage 1 passed: deployment corrections are valid.")
else:
    print("Stage 1 requires investigation before proceeding.")

Mapped grid records: 23400
Mapped records without rainfall: 0
Recalculated district-date rows: 3675
Deployment district-date rows: 3675

Merge status:
_merge
both          3675
left_only        0
right_only       0
Name: count, dtype: int64


,saved_column,recalculated_column,compared_values,missing_comparisons,maximum_absolute_difference,mean_absolute_difference,values_above_tolerance,passed
0,correction_raw_mean_mm,recalculated_raw_mean,3675,0,0.000005,1.420375e-07,0,True
1,global_corrected_mean_mm,recalculated_global_mean,3675,0,0.000005,1.035651e-07,0,True
2,global_corrected_max_mm,recalculated_global_max,3675,0,0.000005,1.388959e-07,0,True
3,regime_corrected_mean_mm,recalculated_regime_mean,3675,0,0.000005,1.047209e-07,0,True
4,regime_corrected_max_mm,recalculated_regime_max,3675,0,0.000005,1.436193e-07,0,True
5,observed_mean_mm,recalculated_observed_mean,3675,0,0.000005,9.123035e-08,0,True
6,observed_max_mm,recalculated_observed_max,3675,0,0.000005,1.312904e-07,0,True
7,grid_cells,recalculated_grid_cells,3675,0,0.000000,0.000000e+00,0,True



VALIDATION RESULT
All district-date rows matched: True
All metric comparisons passed: True
Stage 1 passed: deployment corrections are valid.
